In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.covariance import LedoitWolf
from sklearn.metrics import average_precision_score


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_array = X_cn7.to_numpy()
y_array = y_cn7.to_numpy()

oof_distance = np.zeros(len(X_cn7))


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]

    X_val = X_array[val_idx]


    # 정상 데이터만 사용
    X_train_normal = X_train[y_train == 0]


    # 정상 데이터의 공분산 구조 학습
    covariance_model = LedoitWolf()
    covariance_model.fit(X_train_normal)


    # Validation 데이터의 Mahalanobis Distance
    diff = X_val - covariance_model.location_

    distance = np.sqrt(
        np.sum(
            (diff @ covariance_model.precision_) * diff,
            axis=1
        )
    )

    oof_distance[val_idx] = distance


print(
    "Mahalanobis OOF PR-AUC:",
    average_precision_score(
        y_array,
        oof_distance
    )
)

Mahalanobis OOF PR-AUC: 0.4207075406767129


In [5]:
distance_df = pd.DataFrame({
    "Target": y_array,
    "Mahalanobis_Distance": oof_distance
})

distance_summary = (
    distance_df
    .groupby("Target")["Mahalanobis_Distance"]
    .agg([
        "count",
        "mean",
        "median",
        "min",
        "max"
    ])
)

display(distance_summary)

,count,mean,median,min,max
Target,,,,,
0,1194,4.082752,3.951950,2.264782,28.723267
1,17,14.901747,5.306761,3.998064,38.443124


In [6]:
distance_df["Rank"] = (
    distance_df["Mahalanobis_Distance"]
    .rank(
        ascending=False,
        method="min"
    )
)

display(
    distance_df[
        distance_df["Target"] == 1
    ].sort_values(
        "Mahalanobis_Distance",
        ascending=False
    )
)

,Target,Mahalanobis_Distance,Rank
117,1,38.443124,1.0
116,1,36.037735,2.0
118,1,32.244016,3.0
119,1,32.014420,4.0
120,1,31.688969,5.0
115,1,30.259269,6.0
95,1,6.716049,20.0
111,1,5.797081,28.0
113,1,5.306761,65.0
109,1,5.133638,82.0


In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []

for top_n in [
    5, 6, 10, 15, 20, 25, 30,
    40, 50, 75, 100, 150, 200
]:

    threshold = np.sort(oof_distance)[-top_n]

    y_pred = (
        oof_distance >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array,
        y_pred
    ).ravel()

    results.append({
        "Top N": top_n,
        "Threshold": threshold,
        "Precision": precision_score(
            y_array, y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_array, y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_array, y_pred,
            zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


mahal_threshold_df = pd.DataFrame(results)

display(mahal_threshold_df)

,Top N,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,5,31.688969,1.000000,0.294118,0.454545,5,0,12,1194
1,6,30.259269,1.000000,0.352941,0.521739,6,0,11,1194
2,10,16.133293,0.600000,0.352941,0.444444,6,4,11,1190
3,15,9.314333,0.400000,0.352941,0.375000,6,9,11,1185
4,20,6.716049,0.350000,0.411765,0.378378,7,13,10,1181
5,25,5.945803,0.280000,0.411765,0.333333,7,18,10,1176
6,30,5.692936,0.266667,0.470588,0.340426,8,22,9,1172
7,40,5.579405,0.200000,0.470588,0.280702,8,32,9,1162
8,50,5.420886,0.160000,0.470588,0.238806,8,42,9,1152
9,75,5.166242,0.120000,0.529412,0.195652,9,66,8,1128


### stratifiedKfold oof logistic

In [8]:
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


logistic_oof_prob = cross_val_predict(
    logistic_model,
    X_cn7,
    y_cn7,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [9]:
compare_df = pd.DataFrame({
    "Target": y_cn7.to_numpy(),
    "Logistic_Prob": logistic_oof_prob,
    "Mahalanobis_Distance": oof_distance
})


compare_defect = compare_df[
    compare_df["Target"] == 1
].copy()


compare_defect["Logistic_Rank"] = (
    compare_df["Logistic_Prob"]
    .rank(
        ascending=False,
        method="min"
    )[compare_defect.index]
)


compare_defect["Mahalanobis_Rank"] = (
    compare_df["Mahalanobis_Distance"]
    .rank(
        ascending=False,
        method="min"
    )[compare_defect.index]
)


display(
    compare_defect.sort_values(
        "Mahalanobis_Rank"
    )
)

,Target,Logistic_Prob,Mahalanobis_Distance,Logistic_Rank,Mahalanobis_Rank
117,1,0.901290,38.443124,6.0,1.0
116,1,0.984080,36.037735,1.0,2.0
118,1,0.957725,32.244016,5.0,3.0
119,1,0.958082,32.014420,4.0,4.0
120,1,0.959126,31.688969,3.0,5.0
115,1,0.963258,30.259269,2.0,6.0
95,1,0.080410,6.716049,37.0,20.0
111,1,0.030944,5.797081,94.0,28.0
113,1,0.031087,5.306761,92.0,65.0
109,1,0.146372,5.133638,23.0,82.0
